# 06 · Review Model

**Owner:** Stream B  
**Roadmap:** Stage 7b / 7d  
**Purpose:** Review-helpfulness model: TF-IDF + LR baseline vs. DistilBERT, macro-F1 / PR-AUC.

STUB: not implemented yet. Follow the leakage rules in `CLAUDE.md`.

## Exploration (Stage 1a)

Reads the outputs of `python ml/src/reviews_prepare.py scan` (one `{Category}_summary.json`, `_thin.parquet` and
`_texts.parquet` per category) and turns them into the decisions at the end of this section: the review date window,
the helpful-vote threshold and the categories used. Point it at another folder with the `SCAN_DIR` environment
variable (a relative path is taken from the repo root).

### 1. Config
Finds the repo root (Jupyter runs in the notebook folder, so relative paths won't work), the scan folder and the
figure folder. **`START` / `END` are the review date window** (inclusive months): set them after reading the
vote-vs-date plot in step 4, then re-run everything below it.

In [ ]:
import glob
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score


def find_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "CLAUDE.md").exists():
            return folder
    raise FileNotFoundError(f"No CLAUDE.md in {start} or any parent folder")


ROOT = find_root(Path.cwd().resolve())
SCAN = Path(os.environ.get("SCAN_DIR", ROOT / "ml/data/reviews/scan"))
if not SCAN.is_absolute():
    SCAN = ROOT / SCAN
FIG = ROOT / "ml/artifacts/figures"
FIG.mkdir(parents=True, exist_ok=True)

START, END = "2015-01", "2021-12"  # edit after reading the vote-vs-date plot

# Window as epoch milliseconds: [first ms of START, first ms after END)
START_MS = int(pd.Timestamp(f"{START}-01", tz="UTC").timestamp() * 1000)
END_MS = int((pd.Timestamp(f"{END}-01", tz="UTC") + pd.offsets.MonthBegin(1)).timestamp() * 1000)


def in_window(ts_ms: pd.Series) -> pd.Series:
    return (ts_ms >= START_MS) & (ts_ms < END_MS)


def category_of(path: str, suffix: str) -> str:
    return Path(path).name.removesuffix(suffix)


THIN = sorted(glob.glob(str(SCAN / "*_thin.parquet")))
TEXTS = sorted(glob.glob(str(SCAN / "*_texts.parquet")))
print(f"ROOT   {ROOT}\nSCAN   {SCAN}\nFIG    {FIG}\nwindow {START} .. {END}")

### 2. Load the scan summaries
One line per category: rows, whether the scan finished, any error, malformed lines and the date range.
**Look for:** every category says `completed`, and the data ends around Sep 2023. A `PARTIAL` scan only covers the
start of its file, so its numbers may be biased if the file isn't in random order (see step 9).

In [ ]:
def ms_to_day(ms):
    return pd.to_datetime(ms, unit="ms", utc=True).strftime("%Y-%m-%d") if ms is not None else "n/a"


summaries = {}
for path in sorted(glob.glob(str(SCAN / "*_summary.json"))):
    with open(path, encoding="utf-8") as f:
        s = json.load(f)
    summaries[s["category"]] = s
assert summaries, f"No *_summary.json files in {SCAN}"

for cat, s in summaries.items():
    status = "completed" if s["completed"] else "PARTIAL"
    print(f"{cat:28s} n_rows={s['n_rows']:>12,}  {status:9s}  error={s['error']}  "
          f"n_malformed={s['n_malformed']}  n_negative_votes={s.get('n_negative_votes', 'n/a')}  "
          f"dates={ms_to_day(s['ts_min_ms'])} → {ms_to_day(s['ts_max_ms'])}")

### 3. Helpful-vote distribution
Share of reviews with 0, ≥1, ≥2, ≥3, ≥5 and ≥10 helpful votes, from the exact histogram over every row.
**Look for:** most reviews have 0 votes, so "helpful" (≥3) is a small minority class.

In [ ]:
def vote_counts(hist: dict) -> np.ndarray:
    # index k = exactly k votes for k < 50; index 50 = 50 or more
    return np.array([hist[str(k)] for k in range(50)] + [hist["50+"]], dtype=np.int64)


rows = []
for cat, s in summaries.items():
    c = vote_counts(s["helpful_vote_hist"])
    n = c.sum()
    rows.append({"category": cat, "n": n, "% 0": c[0] / n, "% ≥1": c[1:].sum() / n, "% ≥2": c[2:].sum() / n,
                 "% ≥3": c[3:].sum() / n, "% ≥5": c[5:].sum() / n, "% ≥10": c[10:].sum() / n})
vote_table = pd.DataFrame(rows).set_index("category")
pct = [col for col in vote_table if col.startswith("%")]
vote_table[pct] = (vote_table[pct] * 100).round(2)
vote_table

### 4. Votes vs. review date
Share of reviews with ≥3 helpful votes per calendar quarter (from 2010Q1), per category. The shaded band is the
current `START`..`END` window. **Look for:** the most recent quarters drop because their reviews haven't had time to
collect votes yet. `END` should sit before that drop. Saved to `ml/artifacts/figures/review_votes_by_quarter.png`.

In [ ]:
def months_frame(s: dict) -> pd.DataFrame:
    m = pd.DataFrame.from_dict(s["months"], orient="index")
    m.index = pd.PeriodIndex(m.index, freq="M")
    return m.sort_index()


fig, ax = plt.subplots(figsize=(10, 4.5))
for cat, s in summaries.items():
    m = months_frame(s)
    q = m.groupby(m.index.asfreq("Q")).sum()
    q = q[q.index >= pd.Period("2010Q1", freq="Q")]
    ax.plot(q.index.to_timestamp(), q["v3plus"] / q["n"], marker=".", label=cat)
ax.axvspan(pd.Timestamp(f"{START}-01"), pd.Timestamp(f"{END}-01") + pd.offsets.MonthEnd(1),
           color="grey", alpha=0.12, label=f"window {START}..{END}")
ax.set_ylabel("share of reviews with ≥3 helpful votes")
ax.set_xlabel("review quarter")
ax.set_title("Helpful votes by review date")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "review_votes_by_quarter.png", dpi=150)
plt.show()

### 5. Eligible reviews in the window
Exact counts inside `START`..`END` from the monthly buckets: positives at the ≥3 threshold (`pos_3plus`) and at
≥2 (`pos_2plus`), negatives (`zero` = 0 votes), and the resulting prevalence (share of positives among positives +
negatives). **Look for:** prevalence is the PR-AUC a random guesser would get, so it's the floor every model is
compared against.

In [ ]:
def window_sums(s: dict) -> pd.Series:
    m = months_frame(s)
    m = m[(m.index >= pd.Period(START, freq="M")) & (m.index <= pd.Period(END, freq="M"))]
    return m.sum()


rows = []
for cat, s in summaries.items():
    w = window_sums(s)
    rows.append({"category": cat, "n_window": w.get("n", 0), "pos_3plus": w.get("v3plus", 0),
                 "pos_2plus": w.get("v3plus", 0) + w.get("v2", 0), "zero": w.get("v0", 0)})
eligible = pd.DataFrame(rows).set_index("category").astype("int64")
eligible.loc["TOTAL"] = eligible.sum()
eligible["prev_3plus"] = (eligible["pos_3plus"] / (eligible["pos_3plus"] + eligible["zero"])).round(4)
eligible["prev_2plus"] = (eligible["pos_2plus"] / (eligible["pos_2plus"] + eligible["zero"])).round(4)
eligible

### 6. Length baseline preview (thin sample)
On the thin sample inside the window, keeping only 0-vote and ≥3-vote reviews: how well does review length alone
rank helpful above not helpful? Shows AP (PR-AUC) and ROC-AUC for raw `word_count`, AP for the current stub
`min(1, word_count / 100)`, and the median length per class. **Information only:** don't choose `START`, `END` or the
threshold to make this baseline weaker.

In [ ]:
def length_metrics(t: pd.DataFrame) -> dict:
    y = (t["helpful_vote"] >= 3).to_numpy().astype(int)
    wc = t["word_count"].to_numpy()
    out = {"n": len(t), "prevalence": y.mean() if len(y) else np.nan}
    if len(np.unique(y)) < 2:  # AP / ROC-AUC need both classes
        return {**out, "AP_word_count": np.nan, "ROC_AUC_word_count": np.nan, "AP_stub": np.nan,
                "median_wc_helpful": np.nan, "median_wc_zero": np.nan}
    return {**out,
            "AP_word_count": average_precision_score(y, wc),
            "ROC_AUC_word_count": roc_auc_score(y, wc),
            "AP_stub": average_precision_score(y, np.minimum(1, wc / 100)),
            "median_wc_helpful": np.median(wc[y == 1]),
            "median_wc_zero": np.median(wc[y == 0])}


frames, rows = [], []
for path in THIN:
    t = pd.read_parquet(path, columns=["timestamp_ms", "helpful_vote", "word_count"])
    t = t[in_window(t["timestamp_ms"]) & ((t["helpful_vote"] == 0) | (t["helpful_vote"] >= 3))]
    frames.append(t)
    rows.append({"category": category_of(path, "_thin.parquet"), **length_metrics(t)})
if frames:
    rows.append({"category": "ALL (pooled)", **length_metrics(pd.concat(frames))})
length_table = pd.DataFrame(rows).set_index("category").round(4)
length_table

### 7. Threshold check (text reservoir)
How many reviews with ≥3 votes inside the window are actually available as text in the reservoir (capped per
category), i.e. the positives Stage 4a can train and evaluate on. **Look for:** keep the ≥3 threshold unless the
total is under about 3,000; then consider ≥2.

In [ ]:
rows = []
for path in TEXTS:
    t = pd.read_parquet(path, columns=["timestamp_ms", "helpful_vote"])
    w = in_window(t["timestamp_ms"])
    rows.append({"category": category_of(path, "_texts.parquet"),
                 "texts_3plus_in_window": int((w & (t["helpful_vote"] >= 3)).sum()),
                 "texts_2plus_in_window": int((w & (t["helpful_vote"] >= 2)).sum()),
                 "texts_zero_in_window": int((w & (t["helpful_vote"] == 0)).sum())})
threshold_table = pd.DataFrame(rows).set_index("category")
threshold_table.loc["TOTAL"] = threshold_table.sum()
total_3plus = int(threshold_table.loc["TOTAL", "texts_3plus_in_window"]) if rows else 0
print(f"Total ≥3-vote texts in window: {total_3plus:,} → "
      + ("keep ≥3" if total_3plus >= 3000 else "under ~3,000: consider ≥2"))
threshold_table

### 8. Exact duplicate texts
Share of reservoir texts that have an exact duplicate after `strip()` + lowercase (all copies counted), and the share
that de-duplication would remove (all copies but one). **Look for:** this is a lower bound (near-duplicates aren't
counted, and the reservoir is only a sample); it tells you how much Stage 4a's de-duplication matters.

In [ ]:
rows = []
for path in TEXTS:
    text = pd.read_parquet(path, columns=["text"])["text"]
    norm = text.fillna("").str.strip().str.lower()
    nonempty = norm[norm != ""]
    rows.append({"category": category_of(path, "_texts.parquet"), "n_texts": len(norm),
                 "n_empty": int((norm == "").sum()),
                 "% with a duplicate": round(100 * nonempty.duplicated(keep=False).mean(), 2),
                 "% removed by dedup": round(100 * nonempty.duplicated(keep="first").mean(), 2)})
dup_table = pd.DataFrame(rows).set_index("category")
dup_table

### 9. File-order check
Review date against position in the file (`row_idx`), for every 20th thin-sample row. **Look for:** points rising
from left to right mean the file is sorted by time, so a `PARTIAL` scan covers only the oldest (or newest) reviews.
If the cloud is flat, the order is mixed and a partial scan is roughly representative. Only matters for `PARTIAL`
scans. Saved to `ml/artifacts/figures/review_file_order.png`.

In [ ]:
fig, axes = plt.subplots(1, max(len(THIN), 1), figsize=(5 * max(len(THIN), 1), 4), squeeze=False)
for ax, path in zip(axes[0], THIN):
    t = pd.read_parquet(path, columns=["row_idx", "timestamp_ms"]).iloc[::20]
    ax.scatter(t["row_idx"], pd.to_datetime(t["timestamp_ms"], unit="ms"), s=1, alpha=0.3)
    ax.set_title(category_of(path, "_thin.parquet"))
    ax.set_xlabel("row_idx (position in file)")
    ax.set_ylabel("review date")
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG / "review_file_order.png", dpi=150)
plt.show()

### Decision (fill in)
Copy the numbers from the cells above, then log the decision in `docs/decisions.md` (Labeling definitions →
Review helpfulness label) and `docs/notes/stream-b.md`.

| Item | Value | Reason |
|---|---|---|
| Data end date | | |
| `START` | | |
| `END` | | |
| Helpful threshold (≥3 or ≥2) | | |
| Prevalence per category | | |
| Prevalence, total | | |
| Length AP (raw `word_count`) | | |
| Stub AP (`min(1, wc/100)`) | | |
| Exact-duplicate rate | | |
| Categories used | | |